In [3]:
import torch
import numpy as np
from typing import Tuple, Optional, Union

def compute_kernel_matrix(
    kernel: str,
    gamma: float,
    x: Union[np.ndarray, torch.tensor], 
    y: Union[np.ndarray, torch.tensor],
) -> torch.Tensor:
    """
    Compute the kernel matrix in batch style.
    DOES NOT PERFORM NORMALIZATION

    feat_labeled: (L, D)
    feat_unlabeled: (U, D)
    """

    if kernel == 'rbf':
        return rbf_kernel(x, y, gamma=gamma)
    elif kernel == 'cosine':
        return cosine_similarity(x, y)
    elif kernel == 'linear':
        # plain inner products; diagonal is ||x||^2, NOT 1 — the Vendi code
        # accounts for this via kernel_self_similarity + trace normalization
        return x @ y.T
    else:
        raise ValueError(f"Unknown kernel: {kernel}")

def rbf_kernel(x, y=None, gamma=None):
    """
    x: (N, D)
    y: (M, D)
    gamma: float

    Returns:
        kernel matrix shaped (N, M)
    """
    if y is None:
        y = x
    if gamma is None:
        gamma = 1.0 / x.shape[1]

    x_norm = (x**2).sum(dim=1).view(-1, 1)
    y_norm = (y**2).sum(dim=1).view(1, -1)
    dist = x_norm + y_norm - 2.0 * (x @ y.T)
    return torch.exp(-gamma * dist)

def cosine_similarity(x, y=None):
    """
    x: (N, D)
    y: (M, D)
    """
    if y is None:
        y = x
    # Clamp norms away from zero: a zero-norm feature vector (e.g. a zero
    # gradient under query=gvendi) would otherwise give 0/0 = NaN, which then
    # propagates into the kernel matrix and makes every eigensolver fail. A
    # zero-norm row then has cosine 0 with everything (well-defined).
    nx = torch.norm(x, dim=1, p=2).clamp_min(1e-12).view(-1, 1)
    ny = torch.norm(y, dim=1, p=2).clamp_min(1e-12).view(1, -1)
    return (x @ y.T) / (nx * ny)

In [4]:
feat_labeled = torch.rand(300, 512)
feat_unlabeled = torch.rand(500, 512)
for kernel in ['rbf', 'cosine', 'linear']:
    print(compute_kernel_matrix(kernel, 1.0, feat_unlabeled, feat_labeled).shape)

torch.Size([500, 300])
torch.Size([500, 300])
torch.Size([500, 300])
